# LOAD DATA

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from Bio import pairwise2
from pathlib import Path
DATA_PATH = Path("/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/bpRNA")

In [ ]:
df = pd.read_csv(DATA_PATH / "bpRNA.csv", index_col="id")
df_splits = pd.read_csv(DATA_PATH / "bpRNA_splits.csv", index_col="id")

## Check non canonics nt

In [ ]:
not_valid_ids = []
for i in range(len(df)):
    save = False
    for l in df.iloc[i].sequence:
        if l not in ["A", "C", "G", "U"]: 
            save = True
    if save:
        not_valid_ids.append(df.iloc[i].name)

print(f"count: {len(not_valid_ids)} , Invalid IDs: {not_valid_ids}")

In [ ]:
df.loc[not_valid_ids[0]].sequence

In [ ]:
print("full dataset splits")
display(df_splits.groupby("partition")["fold"].agg("count"))
print("invalid IDs splits")
display(df_splits.loc[not_valid_ids].groupby("partition")["fold"].agg("count"))

In [ ]:
df_splits = df_splits.loc[~df_splits.index.isin(not_valid_ids)].sort_values(["partition", "id"]) 
order_idx = df_splits.index
order_idx

In [ ]:
df_valid = df[~df.index.isin(not_valid_ids)]
df_ordered = df_valid.loc[order_idx]
# df_ordered["sequence"].to_csv(DATA_PATH / "bpRNA_ordered.csv")

# Seqsim

- Igual, tené en cuenta que esto corre en CPU, así que si en un server tenés 32 CPUs, podés correr 32 slices en paralelo
- Mi sugerencia es hacer un script simplemente que por parámetro le pasás de qué fila a qué fila querés que te calcule las distancias, y te deja un h5 con esas filas. Solo tené en cuenta de comenzar en cada fila j, a partir de la columna j+1 hasta el final
- Ese script lo podés lanzar 32 veces en un server, suponiendo que tiene 32 núcleos (en cada server te fijás con htop cuántos núcleos tiene)
- Previo podés hacer una estimación de cuántas similaridades hay en una fila y sabiendo cuánto demora en promedio calcular una similaridad, tenés una idea de cuánto va a demorar un rango de filas (aprox), como para no tirar en un nucleo demasiado

In [ ]:
def get_match_dict(symbols, Match, Mismatch): 
    match_dic = {}
    for simbolA in symbols:
        for simbolB in symbols:
            if simbolA==simbolB:
                match_dic[(simbolA, simbolB)] = Match
            else:
                match_dic[(simbolA, simbolB)] = Mismatch
    return match_dic
    
def compute_similarity(col_seq, row_seq, sim_conf):
    alignm = pairwise2.align.localds(col_seq, row_seq, sim_conf[0], sim_conf[1], sim_conf[2], one_alignment_only=True)[0]
    max_len = max(len(alignm.seqB),len(alignm.seqA))
    IDscore = sum(a==b for a,b in zip(alignm.seqA, alignm.seqB)) / max_len
    return IDscore

def expected_num_comparisons(n, start, end):
    return sum(n - i - 1 for i in range(start, end))

def get_sim_conf():
    Match = 1
    Mismatch = -2
    GapOpen=-5
    GapExtend=-2
    simbolos=["A","C","G","U"]
    match_dic = get_match_dict(simbolos, Match, Mismatch)
    sim_conf = match_dic, GapOpen, GapExtend
    return sim_conf


In [ ]:
seqA = df_ordered.iloc[0+5].sequence
seqB = df_ordered.iloc[1+5].sequence

sim_conf = get_sim_conf()
idscore = compute_similarity(seqA, seqB, sim_conf)
print(f"Similarity: {idscore:.4f}")

In [ ]:
seqA = df_ordered.iloc[0+5].sequence
seqB = df_ordered.iloc[1+5].sequence

sim_conf = get_sim_conf()
idscore = compute_similarity(seqB, seqA, sim_conf)
print(f"Similarity: {idscore:.4f}")

In [ ]:
n = len(df_ordered)
start = 0
end = n
print(f"Expected number of total comparisons: {expected_num_comparisons(n, start, end)}")

In [ ]:
import matplotlib.pyplot as plt
# plot comparison accumulated count for each row 
acc = np.cumsum(n - np.arange(n) - 1)
nparts = 17
parts = [acc.max() * i  / nparts for i in range(1, nparts + 1)]
# divide in equal parts in acc
rows = np.arange(n) 
plt.Figure(figsize=(10, 6))
plt.plot(np.arange(n), acc) 
equal_ids =[]
for part in parts:
    idx = np.searchsorted(acc, part)
    equal_ids.append(idx)
    plt.axvline(x=idx, color='gray', linestyle='--', alpha=0.7, linewidth=0.5)
    plt.text(idx, part, f'{idx:.0f}', color='r', fontsize=8, verticalalignment='bottom', horizontalalignment='right')
    plt.axhline(y=part, color='gray', linestyle='--', alpha=0.7, linewidth=0.5)
plt.xlabel("Row index")
plt.ylabel("Accumulated comparisons")
plt.title("Equal computations per row")

plt.show()

In [ ]:
time37k = 280
time1 = time37k / 37000
time10097780 = time1 * 10097780
time10097780 / 3600 

In [ ]:
chunks = 17
acc = np.cumsum(n - np.arange(n) - 1)
parts = [acc.max() * i  / chunks for i in range(1, chunks + 1)]
equal_ids = [np.searchsorted(acc, part).item() for part in parts]

start = 0
n = len(df_ordered)
for part in equal_ids:
    end = part
    computations = expected_num_comparisons(n, start, end)
    print(f"Expected number of comparisons for rows {start} to {end}: {computations}")
    start = end
equal_ids

In [ ]:
def compute_slice(df, start, end):
    ids = df.index.to_list()
    n = len(df)
    # IDs de las filas que va a calcular este slice
    row_ids = ids[start:end]
    
    sims = pd.DataFrame(
        np.zeros((len(row_ids), n), dtype=np.float32),
        index=row_ids,
        columns=ids,
    )

    total = expected_num_comparisons(n, start, end)
    sim_conf = get_sim_conf()

    with tqdm(total=total, desc=f"Rows {start}:{end}") as pbar:
        for i in range(start, end):
            row_id = ids[i]
            seq_i = df.loc[row_id, "sequence"]

            for j in range(i + 1, n):
                col_id = ids[j]
                seq_j = df.loc[col_id, "sequence"]

                sim = compute_similarity(seq_i, seq_j, sim_conf)

                sims.loc[row_id, col_id] = sim

                pbar.update(1)

    return sims

In [ ]:
start  = 0
end = 2 
n_comp = expected_num_comparisons(n, start, end)

print(f"Dataset size: {n}")
print(f"Rows: [{start}, {end})")
print(f"Comparisons: {n_comp:,}")

# out = compute_slice(df_ordered, start, end)

In [ ]:
def compute_slice(df, start, end):
    ids = df.index.to_list()
    seqs = df["sequence"].astype(str).to_dict()
    sim_conf = get_sim_conf()

    row_ids = ids[start:end]
    sims = pd.DataFrame(0.0, index=row_ids, columns=ids, dtype=np.float32)

    total = expected_num_comparisons(len(ids), start, end)

    with tqdm(total=total, desc=f"Rows {start}:{end}") as pbar:
        for i, row_id in enumerate(ids[start:end], start=start):
            seq_i = seqs[row_id]

            for col_id in ids[i + 1:]:
                sims.loc[row_id, col_id] = compute_similarity(
                    seq_i,
                    seqs[col_id],
                    sim_conf,
                )
                pbar.update()

    return sims

In [ ]:
start  = 0
end = 2 
n_comp = expected_num_comparisons(n, start, end)

print(f"Dataset size: {n}")
print(f"Rows: [{start}, {end})")
print(f"Comparisons: {n_comp:,}")

# out = compute_slice(df_ordered, start, end)

In [ ]:

# out_path = Path(data_path) / f"bprna_seqsim_{start}_{end}.h5"
# out.to_hdf(
#     out_path,
#     key="seqsim",
#     mode="w",
#     format="fixed",
# )

In [ ]:
# sims.to_hdf(DATA_PATH + "bprna_seqsim_f_all.h5", key='rnadist', mode='w')

In [ ]:
REPO_PATH = Path("/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/bpRNA")
OUT_PATH = REPO_PATH / "chunks/chunks"

!ls $OUT_PATH


In [ ]:
from pathlib import Path
import pandas as pd
REPO_PATH = Path("/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/bpRNA")
OUT_PATH = REPO_PATH / "chunks"

chunks = [553,
 1124,
 1714,
 2326,
 2962,
 3625,
 4319,
 5049,
 5820,
 6641,
 7524,
 8483,
 9544,
 10749,
 12178,
 14040,
 18535,
 18537] 

In [ ]:

# CHUNKS = [0, 2, 4, 6, 8]
parts = []
for chunk in chunks:
    file_path = OUT_PATH / f"bprna_seqsim_{str(chunk)}.h5"
    parts.append(pd.read_hdf(file_path, key="seqsim"))

seqsim = pd.concat(parts, axis=0)

In [ ]:
arr = seqsim.to_numpy(dtype=np.float32)

seqsim_full = pd.DataFrame(
    arr + arr.T,
    index=seqsim.index,
    columns=seqsim.columns,
    dtype=np.float32,
)

np.fill_diagonal(seqsim_full.values, 1.0)
seqsim_full

In [ ]:
# save full similarity matrix
seqsim_full.to_hdf(OUT_PATH / "bprna_seqsim_full.h5", key="seqsim", mode="w", format="fixed")

# check seqsim

In [ ]:

import numpy as np
import pandas as pd
from tqdm import tqdm
from Bio import pairwise2
from pathlib import Path

In [ ]:
def get_match_dict(symbols, Match, Mismatch): 
    match_dic = {}
    for simbolA in symbols:
        for simbolB in symbols:
            if simbolA==simbolB:
                match_dic[(simbolA, simbolB)] = Match
            else:
                match_dic[(simbolA, simbolB)] = Mismatch
    return match_dic
    
def compute_similarity(col_seq, row_seq, sim_conf):
    alignm = pairwise2.align.localds(col_seq, row_seq, sim_conf[0], sim_conf[1], sim_conf[2], one_alignment_only=True)[0]
    max_len = max(len(alignm.seqB),len(alignm.seqA))
    IDscore = sum(a==b for a,b in zip(alignm.seqA, alignm.seqB)) / max_len
    return IDscore

def expected_num_comparisons(n, start, end):
    return sum(n - i - 1 for i in range(start, end))

def get_sim_conf():
    Match = 1
    Mismatch = -2
    GapOpen=-5
    GapExtend=-2
    simbolos=["A","C","G","U"]
    match_dic = get_match_dict(simbolos, Match, Mismatch)
    sim_conf = match_dic, GapOpen, GapExtend
    return sim_conf


In [ ]:

DATA_PATH = Path("/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/bpRNA")

seqsim_full = pd.read_hdf(DATA_PATH / "chunks"/"bprna_seqsim_full.h5", key="seqsim") 

In [ ]:
seqsim_full.head(5)

In [ ]:
id1, id2 =seqsim_full.index[0], seqsim_full.index[0]
dataset = pd.read_csv(DATA_PATH / "bpRNA_ordered.csv", index_col="id")
seq1, seq2 = dataset.loc[id1, "sequence"], dataset.loc[id2, "sequence"]
compute_similarity(seq1, seq2, get_sim_conf())